In [1]:
import pandas as pd
import mysql.connector 

connection = mysql.connector.connect(
    host = "Localhost",
    user = "root",
    password = "12345",
    database="market_analysis"
)
cursor =  connection.cursor()
cursor


# Which customer segments have the highest response rate to campaigns?

In [ ]:
query="""SELECT Income_band, Age_band, Web_engagement_band, Family_band, Spend_segment, Customer_Count,
   
    ROUND(Campaign1_Accepted * 100.0 / Customer_Count, 2) AS Campaign1_Response_Rate,
    ROUND(Campaign2_Accepted * 100.0 / Customer_Count, 2) AS Campaign2_Response_Rate,
    ROUND(Campaign3_Accepted * 100.0 / Customer_Count, 2) AS Campaign3_Response_Rate,
    ROUND(Campaign4_Accepted * 100.0 / Customer_Count, 2) AS Campaign4_Response_Rate,
    ROUND(Campaign5_Accepted * 100.0 / Customer_Count, 2) AS Campaign5_Response_Rate,
    ROUND(Latest_Campaign_Accepted * 100.0 / Customer_Count, 2) AS Latest_Campaign_Response_Rate

FROM sql_segmentation_table

ORDER BY Campaign2_Response_Rate DESC limit 5"""
#change the campaign number in order by to view for each campaign
df=pd.read_sql(query, connection)
df



C:\Users\hamsa\AppData\Local\Temp\ipykernel_17552\4161079146.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,Income_band,Age_band,Web_engagement_band,Family_band,Spend_segment,Customer_Count,Campaign1_Response_Rate,Campaign2_Response_Rate,Campaign3_Response_Rate,Campaign4_Response_Rate,Campaign5_Response_Rate,Latest_Campaign_Response_Rate
0,High Income,Senior,Low engagement,Parent of a teen,Low spender,16,18.75,18.75,12.50,0.00,6.25,18.75
1,High Income,Young,Average engagement,Parent of child & teen,High spender,9,11.11,11.11,11.11,11.11,0.00,11.11
2,Middle Income,Senior,High engagement,Parent of a teen,High spender,24,12.50,8.33,8.33,8.33,8.33,45.83
3,Middle Income,Senior,Average engagement,Parent of a child,High spender,55,20.00,7.27,1.82,7.27,7.27,16.36
4,High Income,Senior,Low engagement,Parent of a child,High spender,28,10.71,7.14,3.57,3.57,7.14,21.43


# How do spending patterns across products (wine, fruits, meat, fish, sweets, gold) vary by age, income, marital status, and country?

In [ ]:
query="""SELECT
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_band,
    
    CASE
        WHEN d.Age<45 THEN 'Young'
        When d.Age BETWEEN 45 and 63 THEN 'Middle aged'
        ELSE 'Senior'
    END AS Age_band,

    m.Marital_Status,

    m.Country,

    COUNT(*) AS Customer_Count,

    ROUND(AVG(m.MntWines), 2) AS Avg_Wine_Spend,
    ROUND(AVG(m.MntFruits), 2) AS Avg_Fruit_Spend,
    ROUND(AVG(m.MntMeatProducts), 2) AS Avg_Meat_Spend,
    ROUND(AVG(m.MntFishProducts), 2) AS Avg_Fish_Spend,
    ROUND(AVG(m.MntSweetProducts), 2) AS Avg_Sweets_Spend,
    ROUND(AVG(m.MntGoldProds), 2) AS Avg_Gold_Spend,

    ROUND(AVG(m.MntWines + m.MntFruits + m.MntMeatProducts + m.MntFishProducts + m.MntSweetProducts + m.MntGoldProds), 2) AS Avg_Total_Spend

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID

    GROUP BY
        Income_Band,
        Age_Band,
        m.Marital_Status,
        m.Country

    ORDER BY
    Avg_Total_Spend DESC limit 20"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_17552\3169197076.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,Income_band,Age_band,Marital_Status,Country,Customer_Count,Avg_Wine_Spend,Avg_Fruit_Spend,Avg_Meat_Spend,Avg_Fish_Spend,Avg_Sweets_Spend,Avg_Gold_Spend,Avg_Total_Spend
0,High Income,Young,Widow,Mexico,1,883.00,0.00,1032.00,259.00,135.00,1.00,2310.00
1,High Income,Senior,Widow,Mexico,1,992.00,19.00,1023.00,259.00,6.00,9.00,2308.00
2,High Income,Young,YOLO,Australia,1,1078.00,117.00,398.00,257.00,6.00,137.00,1993.00
3,High Income,Senior,Absurd,Saudi Arabia,2,1294.00,10.50,359.00,43.50,4.00,89.00,1800.00
4,Middle Income,Middle aged,Widow,Mexico,2,864.00,5.00,755.00,70.50,2.00,32.00,1728.50
5,High Income,Senior,Absurd,Mexico,2,749.00,58.50,750.50,59.50,47.50,42.00,1707.00
6,High Income,Middle aged,Widow,Mexico,5,599.60,48.40,769.60,91.00,89.40,40.80,1638.80
7,High Income,Senior,YOLO,Germany,5,592.60,33.80,820.20,88.20,36.80,54.60,1626.20
8,High Income,Senior,Absurd,India,7,975.43,63.29,373.43,95.00,67.00,42.00,1616.14
9,Middle Income,Young,Alone,India,2,696.00,71.50,602.50,49.50,11.00,69.00,1499.50


# Which channels (web, store, catalog, deals) are most used by high-value customers, and how often do they visit the website?

In [8]:
query="""SELECT
    CASE
        WHEN d.Total_Spend < 441 THEN 'Low Value'
        WHEN d.Total_Spend < 1600 THEN 'Medium Value'
        ELSE 'High Value'
    END AS Customer_Value,

    COUNT(*) AS Customer_Count,

    ROUND(AVG(m.NumWebPurchases), 2) AS Avg_Web_Purchases,
    ROUND(AVG(m.NumStorePurchases), 2) AS Avg_Store_Purchases,
    ROUND(AVG(m.NumCatalogPurchases), 2) AS Avg_Catalog_Purchases,
    ROUND(AVG(m.NumDealsPurchases), 2) AS Avg_Deals_Purchases,
    ROUND(AVG(m.NumWebVisitsMonth), 2) AS Avg_Website_Visits

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID

    GROUP BY Customer_Value

    ORDER BY
    CASE Customer_Value
        WHEN 'High Value' THEN 1
        WHEN 'Medium Value' THEN 2
        ELSE 3
    END"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_17552\1260752853.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,Customer_Value,Customer_Count,Avg_Web_Purchases,Avg_Store_Purchases,Avg_Catalog_Purchases,Avg_Deals_Purchases,Avg_Website_Visits
0,High Value,5213,5.46,6.32,3.22,1.98,4.02
1,Medium Value,22821,4.93,5.52,2.77,2.12,4.65
2,Low Value,27952,3.46,3.74,1.36,2.25,5.81


# Are there specific segments that are under-served (low spending, high visits, low response)?

In [10]:
query="""SELECT Income_band, Age_band, Web_engagement_band, Family_band, Spend_segment, Customer_Count,

    ROUND((Campaign1_Accepted + Campaign2_Accepted + Campaign3_Accepted + Campaign4_Accepted + Campaign5_Accepted +
            Latest_Campaign_Accepted) * 100.0 / (Customer_Count * 6), 2) AS Overall_Response_Rate

FROM sql_segmentation_table

WHERE Spend_segment = 'Low spender'
  AND Web_engagement_band = 'High engagement'

ORDER BY Overall_Response_Rate ASC"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_17552\2486487363.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,Income_band,Age_band,Web_engagement_band,Family_band,Spend_segment,Customer_Count,Overall_Response_Rate
0,Low Income,Young,High engagement,Parent of child & teen,Low spender,439,1.67
1,Low Income,Young,High engagement,Parent of a child,Low spender,681,1.71
2,Low Income,Middle aged,High engagement,Parent of a child,Low spender,1279,1.93
3,Low Income,Senior,High engagement,Parent of a child,Low spender,310,2.04
4,Low Income,Senior,High engagement,Parent of child & teen,Low spender,151,2.32
5,Low Income,Middle aged,High engagement,Parent of child & teen,Low spender,723,2.33
6,Middle Income,Young,High engagement,Parent of child & teen,Low spender,295,3.28
7,Low Income,Middle aged,High engagement,Parent of a teen,Low spender,58,3.45
8,Low Income,Middle aged,High engagement,Adult,Low spender,164,3.66
9,Low Income,Young,High engagement,Parent of a teen,Low spender,29,4.02


# What are characteristics of “ideal target customers” for future campaigns? (age range, income band, family composition, country, etc.)

In [13]:
query="""SELECT
    CASE
        WHEN m.Income < 28240.225 THEN 'Low Income'
        WHEN m.Income <= 86917.575 THEN 'Middle Income'
        ELSE 'High Income'
    END AS Income_Band,

    CASE
        WHEN d.Age < 45 THEN 'Young'
        WHEN d.Age <= 63 THEN 'Middle-aged'
        ELSE 'Senior'
    END AS Age_Band,

    CASE
        WHEN m.Kidhome >= 1 AND m.Teenhome >= 1 THEN 'Parent of child & teen'
        WHEN m.Kidhome >= 1 THEN 'Parent of a child'
        WHEN m.Teenhome >= 1 THEN 'Parent of a teen'
        ELSE 'Adult'
    END AS Family_Band,

    m.Marital_Status,

    m.Country,

    COUNT(*) AS Customer_Count,

    ROUND((SUM(m.AcceptedCmp1) + SUM(m.AcceptedCmp2) + SUM(m.AcceptedCmp3) + SUM(m.AcceptedCmp4) + SUM(m.AcceptedCmp5) + 
        SUM(m.Response)) * 100.0 / (COUNT(*) * 6), 2) AS Overall_Response_Rate,

    ROUND(AVG(d.Total_Spend), 2) AS Avg_Total_Spend

    FROM marketing_campaigns_data m
    JOIN derived_table d
    ON m.ID = d.ID

    GROUP BY
        Income_Band,
        Age_Band,
        Family_Band,
        m.Marital_Status,
        m.Country

    HAVING COUNT(*) >= 10

    ORDER BY
        Overall_Response_Rate DESC,
        Avg_Total_Spend DESC limit 30"""
df=pd.read_sql(query, connection)
df

C:\Users\hamsa\AppData\Local\Temp\ipykernel_17552\3931380194.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, connection)


,Income_Band,Age_Band,Family_Band,Marital_Status,Country,Customer_Count,Overall_Response_Rate,Avg_Total_Spend
0,High Income,Middle-aged,Adult,Alone,Canada,13,29.49,1045.46
1,High Income,Middle-aged,Adult,YOLO,Spain,17,23.53,875.18
2,High Income,Middle-aged,Parent of a teen,Widow,Australia,12,20.83,1313.75
3,High Income,Middle-aged,Parent of a child,Divorced,USA,11,19.70,663.82
4,Middle Income,Senior,Parent of a child,Widow,Canada,11,19.70,579.64
5,High Income,Senior,Parent of a teen,Together,India,19,19.30,1321.68
6,High Income,Young,Adult,Married,USA,13,19.23,1237.62
7,High Income,Senior,Parent of a teen,Divorced,Australia,21,19.05,1305.48
8,High Income,Senior,Parent of a teen,Married,Germany,10,18.33,1317.30
9,High Income,Young,Adult,Single,USA,10,18.33,734.60
